# Demo: Document Loading & Chunking Strategies
### Module 5, Topic 2 — RAG from Scratch

**What you'll see in this notebook:**
1. Load a sample document (the Naija One Bank policy manual from Topic 1, extended)
2. Split it with naive fixed-size chunking
3. Split it with fixed-size chunking + overlap
4. Split it with sentence-aware chunking
5. Compare all three side by side on the same passage

By the end, you'll see exactly why chunking strategy matters — using the same document each time so the differences are easy to spot.


## Step 1 — Load the Sample Document

In a real system, this text would be extracted from a PDF or Word file (Step 0 of "loading"). Here, it's already plain text — this is what loading hands off to chunking.

In [ ]:
document = """Naija One Bank — Flexi Save Account Policy (Effective 2026)

The Flexi Save account is Naija One Bank's flagship savings product for individual customers. It is designed for customers who want easy access to their funds while still earning competitive interest. The account has no monthly maintenance fee as long as the minimum balance is maintained.

Interest is calculated daily and credited monthly at a rate of 4.2% per annum. The minimum opening balance required to activate the account is NGN 5,000. To continue earning interest, customers must maintain a minimum balance of NGN 1,000 at all times.

Customers are permitted 3 free withdrawals per month. A fee of NGN 500 applies to each withdrawal beyond this limit. Withdrawals can be made via the mobile app, at any branch, or through an ATM using the Flexi Save debit card.

Accounts that fall below the minimum balance for more than 60 consecutive days will be automatically converted to a Basic Save account, which does not earn interest. Customers can reactivate Flexi Save status by restoring the minimum balance."""

print(f"Document length: {len(document)} characters")
print(document)

## Step 2 — Naive Fixed-Size Chunking

The simplest possible approach: cut the text every N characters, no matter what's at that position.

In [ ]:
def fixed_size_chunks(text, chunk_size):
    chunks = []
    for start in range(0, len(text), chunk_size):
        chunks.append(text[start:start + chunk_size])
    return chunks

fixed_chunks = fixed_size_chunks(document, chunk_size=140)

print(f"Number of chunks: {len(fixed_chunks)}\n")
for i, chunk in enumerate(fixed_chunks):
    print(f"--- Chunk {i+1} ---")
    print(chunk)
    print()

## Step 3 — Look for the Damage

Scan the chunks above for cuts that land in the middle of a sentence, a number, or a word. Fixed-size chunking has no idea what it's cutting through — it only counts characters.

**Look specifically at the chunk containing "4.2% per annum" — is the full sentence about when interest is credited still together, or did it get split?**

## Step 4 — Fixed-Size Chunking With Overlap

Now we add overlap: each new chunk starts a little *before* the previous one ended, so content near a boundary appears in both chunks.

In [ ]:
def overlapping_chunks(text, chunk_size, overlap):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

overlap_chunks_list = overlapping_chunks(document, chunk_size=140, overlap=40)

print(f"Number of chunks: {len(overlap_chunks_list)}\n")
for i, chunk in enumerate(overlap_chunks_list):
    print(f"--- Chunk {i+1} ---")
    print(chunk)
    print()

## Step 5 — Notice What Changed

There are now more chunks than in Step 2, and you'll see repeated text at the start/end of neighbouring chunks. That repetition is the safety net — a fact that got cut off at the edge of one chunk is likely to appear whole in the next one.

It's still not perfect: individual sentences can still be split. Overlap reduces the damage from fixed-size cuts, but it doesn't fix the root problem — the cuts still ignore sentence boundaries entirely.

## Step 6 — Sentence-Aware Chunking

This time, we split on sentence boundaries first, then group whole sentences together up to a target size — so no chunk ever ends mid-sentence.

In [ ]:
import re

def split_into_sentences(text):
    # Simple sentence splitter: splits after '.', '!', or '?' followed by a space
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    return [s for s in sentences if s]

def sentence_aware_chunks(text, target_size):
    sentences = split_into_sentences(text)
    chunks = []
    current_chunk = ""

    for sentence in sentences:
        if len(current_chunk) + len(sentence) <= target_size or not current_chunk:
            current_chunk += (" " if current_chunk else "") + sentence
        else:
            chunks.append(current_chunk)
            current_chunk = sentence

    if current_chunk:
        chunks.append(current_chunk)

    return chunks

sentence_chunks = sentence_aware_chunks(document, target_size=140)

print(f"Number of chunks: {len(sentence_chunks)}\n")
for i, chunk in enumerate(sentence_chunks):
    print(f"--- Chunk {i+1} ---")
    print(chunk)
    print()

## Step 7 — Check the Boundaries

Every chunk above should end with a complete sentence — none should trail off mid-word or mid-number. Chunk sizes are less even than Step 2 or Step 4, but every chunk is a coherent, complete thought.

## Step 8 — Side-by-Side Comparison

Let's isolate one specific fact — the interest rate sentence — and see how each strategy handled it.

In [ ]:
target_phrase = "4.2% per annum"

print("FIXED-SIZE CHUNKING:")
for i, chunk in enumerate(fixed_chunks):
    if target_phrase in chunk:
        print(f"  Found whole in chunk {i+1}: ...{chunk.strip()}...")
        break
else:
    print("  NOT found whole in any single chunk — it was split across chunks!")

print("\nFIXED-SIZE + OVERLAP:")
for i, chunk in enumerate(overlap_chunks_list):
    if target_phrase in chunk:
        print(f"  Found whole in chunk {i+1}: ...{chunk.strip()}...")
        break
else:
    print("  NOT found whole in any single chunk!")

print("\nSENTENCE-AWARE:")
for i, chunk in enumerate(sentence_chunks):
    if target_phrase in chunk:
        print(f"  Found whole in chunk {i+1}: ...{chunk.strip()}...")
        break
else:
    print("  NOT found whole in any single chunk!")

## Step 9 — What This Means for Retrieval

If a retriever later searches for "what interest rate does Flexi Save earn?", it can only return whatever text lives inside a single chunk. If the rate and the sentence explaining it were split apart by a naive fixed-size cut, the retriever might return a chunk with the number but no context — or context with no number.

Sentence-aware chunking (with a bit of overlap added on top, in a real system) gives the retriever in Topic 3 the best possible chance of returning one clean, complete, useful chunk.

## What's Next

Now that the document is split into well-formed chunks, Topic 3 builds a **retriever** — the piece that takes a question and automatically finds which of these chunks is actually relevant, instead of us searching through them by eye like we just did.